# Next Ravi Varma, M3 demo

Runs the **default pipeline** (c1: raw text + the `rrv_v1` LoRA, `docs/TEAM_DECISIONS_M3.md` D1) on three scenes, checks
that the same seed gives the same image, shows the LoRA's effect, runs the CLI once, and shows one KB-based mode.

**Runtime:** Colab, **T4 GPU** (Runtime → Change runtime type). About 5 minutes in total, mostly the first model load.
**Needs:** the M2 LoRA on your Drive at the path in `config/m3.json` (see the README), and a classic GitHub token with
`repo` scope (the repo is private).

## 1. Setup

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import getpass, os, subprocess
if not os.path.exists("/content/m3"):
    tok = getpass.getpass("GitHub token (classic, 'repo' scope): ").strip()
    subprocess.run(["git", "clone", "-q", f"https://{tok}@github.com/ananyashetty-bits/next-ravi-varma-m3", "/content/m3"], check=True)
    del tok
%cd /content/m3
!git rev-parse --short HEAD
!bash pipeline/setup_session.sh gen

In [ ]:
import json, sys, time
sys.path.insert(0, "/content/m3")
from pipeline.generate_api import text_to_image, generate, release, CFG
from IPython.display import display
from PIL import Image
lora = CFG["models"][CFG["default_model"]]["lora_path"]
assert os.path.exists(lora), f"LoRA not found at {lora}: see the README's Drive prerequisites"
print("model:", CFG["default_model"], "| sampler", CFG["sampler"], CFG["steps"], "steps, CFG", CFG["cfg"], "|", CFG["size"], "px")

def row(images, size=320):
    out = Image.new("RGB", (size * len(images), size), "white")
    for i, im in enumerate(images):
        out.paste(im.resize((size, size)), (i * size, 0))
    return out

## 2. Three scenes with the default pipeline

In [ ]:
SCENES = [
    "Sita sitting alone under a tree in the Ashoka grove, sorrowful, the garden quiet at dusk",
    "Hanuman kneeling before Rama and offering him Sita's ring, Rama reaching out to take it",
    "Krishna playing the flute under a kadamba tree while cows graze around him",
]
images = []
for text in SCENES:
    t0 = time.time()
    img, exp, scene = text_to_image(text, seed=1)          # mode="raw" is the default
    images.append(img)
    print(f"{time.time() - t0:5.1f} s | characters: {[c['canonical'] for c in scene['characters']]}")
    print("        prompt:", exp["positive"])
display(row(images))

## 3. Same seed, same image

In [ ]:
import numpy as np
again, _, _ = text_to_image(SCENES[0], seed=1)
diff = np.abs(np.asarray(again, dtype=int) - np.asarray(images[0], dtype=int)).max()
print("max pixel difference vs the first run:", diff, "(0 = identical)")
print("reproducible" if diff == 0 else "NOT identical: check that nothing else changed (model, mode, seed)")

## 4. What the LoRA adds: base model vs `rrv_v1` (same prompt, same seed)

In [ ]:
text = SCENES[1]
base, _, _ = text_to_image(text, seed=1, model="base")
lora_img, _, _ = text_to_image(text, seed=1)
display(row([base, lora_img]))
print("left: base SD1.5 | right: rrv_v1 (default)")

## 5. The CLI (same pipeline, one command)

In [ ]:
release()      # the CLI is a separate process and loads its own copy of the model
!python pipeline/cli.py generate "Yashoda churning butter in a large pot in the courtyard of her house" --seed 2 --out /content/cli_demo.png
display(Image.open("/content/cli_demo.png").resize((320, 320)))

## 6. A KB-based mode (optional)

`mode="template"` (c2) builds the prompt from the parsed scene and the KB's character descriptions, with no LLM.
It was **less faithful than raw text** in the M3 evaluation (`eval/EVAL_REPORT.md`), so it is shown only for
comparison. The LLM modes (`llm_kb`, `llm_nokb`) also work here, but they load a 4B LLM first (a few minutes).

In [ ]:
img, exp, scene = text_to_image(SCENES[1], seed=1, mode="template")
print("prompt:", exp["positive"], f"({exp['token_count']} tokens)")
display(row([images[1], img]))
print("left: raw (default) | right: template")